# 일별 댓글 수급 신호 LLM 보고서 검수

`llm_report`에 적재된 보고서를 읽고 계약대로 저장됐는지 확인한다.

2026-08-07 개편으로 보고서 구조가 바뀌었다.

- `prompt_version` `market_commentary_v7` -> `market_commentary_v8`
- `report_schema_version` `4` -> `5`
- `evidence_schema_version` `2` -> `3`

기여 키워드와 대표 댓글 근거는 제거됐고 정형 수치 근거로 대체됐다.
따라서 `key_expressions`, `used_comment_refs`, `representative_comments`,
`details.evidence`의 키워드 목록은 더 이상 존재하지 않는다.

버전을 지정하지 않으면 최신 계약(v8)을 조회한다.


In [1]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent.parent.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))


In [2]:
from pilos.storage.db import get_engine

engine = get_engine()


## 1. 적재 현황 확인

조회 결과가 비어 있을 때 원인을 먼저 구분한다.

- 보고서 자체가 없는 경우
- 다른 `prompt_version`으로만 적재된 경우


In [3]:
import pandas as pd

from sqlalchemy import text

inventory_sql = text(
    """
    SELECT
        prompt_version,
        report_schema_version,
        evidence_schema_version,
        status,
        COUNT(*) AS report_count,
        MIN(model_date) AS first_model_date,
        MAX(model_date) AS last_model_date
    FROM llm_report
    GROUP BY
        prompt_version,
        report_schema_version,
        evidence_schema_version,
        status
    ORDER BY
        prompt_version,
        report_schema_version,
        status
    """
)

with engine.connect() as conn:
    inventory_df = pd.DataFrame(
        [dict(row) for row in conn.execute(inventory_sql).mappings()]
    )

inventory_df


,prompt_version,report_schema_version,evidence_schema_version,status,report_count,first_model_date,last_model_date
0,llm_report_v2,2,2,ready,1,2026-07-30,2026-07-30
1,llm_report_v3,2,2,ready,23,2026-07-27,2026-07-31
2,market_commentary_v10,5,3,insufficient_evidence,10,2026-07-27,2026-07-27
3,market_commentary_v10,5,3,ready,17,2026-07-28,2026-07-30
4,market_commentary_v11,5,3,insufficient_evidence,10,2026-07-27,2026-07-27
5,market_commentary_v11,5,3,ready,59,2026-07-28,2026-08-06
6,market_commentary_v12,5,3,insufficient_evidence,10,2026-07-27,2026-07-27
7,market_commentary_v12,5,3,ready,71,2026-07-28,2026-08-06
8,market_commentary_v13,5,3,insufficient_evidence,11,2026-07-27,2026-07-29
9,market_commentary_v13,5,3,ready,79,2026-07-28,2026-08-06


## 2. 검수 대상 조회

`REPORT_VERSION`을 바꾸면 이전 버전도 확인할 수 있다. 다만 아래 검수
셀은 v8 구조를 기준으로 작성했으므로 이전 버전에는 없는 컬럼이 생긴다.


In [4]:
from pilos.dto.llm_report_dto import (
    EVIDENCE_SCHEMA_VERSION,
    PROMPT_VERSION,
    REPORT_SCHEMA_VERSION,
)

# 코드 상수를 그대로 사용하여 버전이 올라가도 노트북이 따라간다.
REPORT_VERSION = PROMPT_VERSION

print(
    f"prompt_version={PROMPT_VERSION} "
    f"report_schema_version={REPORT_SCHEMA_VERSION} "
    f"evidence_schema_version={EVIDENCE_SCHEMA_VERSION}"
)

report_sql = text(
    """
    SELECT *
    FROM llm_report
    WHERE prompt_version = :version
    ORDER BY model_date DESC, stock_id ASC
    """
)

with engine.connect() as conn:
    records = [
        dict(row)
        for row in conn.execute(
            report_sql,
            {"version": REPORT_VERSION},
        ).mappings()
    ]

print(f"{REPORT_VERSION} 보고서 {len(records)}건")

if not records:
    print(
        "조회 결과가 없습니다. 위 적재 현황에서 실제 prompt_version을 "
        "확인하거나 pilos.jobs.generate_llm_reports를 먼저 실행하세요."
    )


prompt_version=market_commentary_v13 report_schema_version=5 evidence_schema_version=3
market_commentary_v13 보고서 90건


## 3. report_json 펼치기

In [5]:
import json

if not records:
    raise SystemExit("검수할 보고서가 없습니다.")


base_df = pd.DataFrame(records)


def parse_report_json(value: str | bytes | dict) -> dict:
    """DB 드라이버가 JSON 컬럼을 어떤 형태로 주더라도 dict로 만든다."""
    if isinstance(value, dict):
        return value

    if isinstance(value, (bytes, bytearray)):
        value = value.decode("utf-8")

    if isinstance(value, str):
        return json.loads(value)

    raise TypeError(f"report_json의 타입이 올바르지 않습니다: {type(value)}")


report_series = base_df["report_json"].map(parse_report_json)

report_df = pd.json_normalize(report_series.tolist(), sep=".")

full_df = pd.concat(
    [
        base_df.drop(columns=["report_json"]).reset_index(drop=True),
        report_df.add_prefix("report.").reset_index(drop=True),
    ],
    axis=1,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

sorted(full_df.columns)


['created_at',
 'daily_document_id',
 'evidence_schema_version',
 'input_hash',
 'input_tokens',
 'llm_report_id',
 'model',
 'model_date',
 'negative_result_id',
 'output_tokens',
 'positive_result_id',
 'prompt_version',
 'provider',
 'provider_response_id',
 'report.actual_supply_index',
 'report.comment_count',
 'report.comment_signal_score',
 'report.commentary_source',
 'report.conclusion',
 'report.daily_document_id',
 'report.details.active_model_variant',
 'report.details.artifact_schema_version',
 'report.details.calibration_schema_version',
 'report.details.evidence.actual_supply_index',
 'report.details.evidence.comment_count',
 'report.details.evidence.comment_signal_score',
 'report.details.evidence.previous_signal_score',
 'report.details.evidence.signal_change',
 'report.details.evidence.signal_level',
 'report.details.evidence.signal_ma5',
 'report.details.evidence.signal_status',
 'report.details.evidence.supply_direction',
 'report.details.inference_status',
 'report

## 4. 신호 중심 검수표

`commentary_source`로 LLM이 작성한 보고서와 deterministic 요약을
구분한다.


In [6]:
REVIEW_COLUMNS = {
    "llm_report_id": "보고서ID",
    "report.model_date": "기준일",
    "report.stock_code": "종목코드",
    "report.stock_name": "종목명",
    "report.supply_direction": "수급방향",
    "report.actual_supply_index": "실제수급지수",
    "report.comment_signal_score": "댓글신호",
    "report.signal_level": "신호강도",
    "report.signal_status": "신호상태",
    "report.previous_signal_score": "전일신호",
    "report.signal_change": "전일대비",
    "report.signal_ma5": "직전5일평균",
    "report.comment_count": "댓글수",
    "report.commentary_source": "작성주체",
    "status": "저장상태",
    "report.market_commentary": "시장코멘터리",
    "report.conclusion": "한줄정리",
    "report.prompt_version": "프롬프트버전",
    "provider": "공급자",
    "model": "모델",
}

available = [column for column in REVIEW_COLUMNS if column in full_df.columns]
missing = [column for column in REVIEW_COLUMNS if column not in full_df.columns]

if missing:
    print(f"이 버전에 없는 컬럼: {missing}")

review_df = (
    full_df[available]
    .rename(columns=REVIEW_COLUMNS)
    .sort_values(by=["기준일", "종목명"], ascending=[False, True])
    .reset_index(drop=True)
)

review_df


,보고서ID,기준일,종목코드,종목명,수급방향,실제수급지수,댓글신호,신호강도,신호상태,전일신호,전일대비,직전5일평균,댓글수,작성주체,저장상태,시장코멘터리,한줄정리,프롬프트버전,공급자,모델
0,656,2026-08-06,373220,LG에너지솔루션,SELL,-0.532216,70,높음,ready,83.0,-13.0,80.0,9,llm,ready,"현재는 개인투자자의 매도가 더 많습니다. 댓글 수급 신호는 70점으로 높은 편이지만, 어제와 비교하면 13점 낮아졌습니다. 또한 최근 5거래일 평균인 80점과 비교해도 현재가 더 낮은 상태입니다.","현재 개인투자자의 매도가 더 많은 상태이며, 댓글 신호는 70점으로 높은 편이나 어제보다도 최근 5거래일 평균보다 낮은 수준입니다.",market_commentary_v13,academy,qwen3.5-9b
1,655,2026-08-06,035420,NAVER,SELL,-0.150678,21,낮음,ready,65.0,-44.0,51.0,151,llm,ready,"현재는 개인투자자의 매도가 더 많습니다. 댓글 수급 신호가 21점으로 낮은 편이며, 이는 최근 5거래일 평균인 51점보다도 낮습니다. 다만 어제와 비교하면 44점이 크게 떨어졌습니다.","NAVER의 현재 댓글 수급 신호는 개인투자자의 매도가 우세한 21점으로, 직전 거래일의 65점 대비 44점 하락하고 최근 5일 평균인 51점에도 미치지 못하는 낮은 상태입니다.",market_commentary_v13,academy,qwen3.5-9b
2,652,2026-08-06,000660,SK하이닉스,BUY,0.439048,75,높음,ready,53.0,22.0,59.0,9702,llm,ready,"현재는 개인투자자의 매수가 더 많습니다. 오늘 댓글 수급 신호는 75점으로 높은 편인데, 이는 직전 거래일의 53점보다 22점 높아졌습니다. 다만, 이 점수는 최근 5거래일 평균인 59점보다도 높습니다.","SK하이닉스는 개인투자자의 매수가 우세한 가운데 댓글 수급 신호가 75점으로 높은 편이며, 어제보다도 최근 5일 평균보다 더 높은 상태입니다.",market_commentary_v13,academy,qwen3.5-9b
3,661,2026-08-06,034020,두산에너빌리티,BUY,0.062034,1,매우 낮음,ready,70.0,-69.0,48.0,231,llm,ready,"두산에너빌리티의 현재 수급은 개인투자자가 더 많이 매수하고 있습니다. 하지만 오늘 댓글 신호 점수는 1점으로 매우 낮은 편이며, 이는 어제(70점) 대비 69점이 줄어든 결과입니다. 또한 최근 5거래일 평균인 48점과 비교해도 낮습니다.","현재 개인투자자의 매수가 우세한 상태이나, 댓글 신호는 어제보다 69점 낮아져 최근 5일 평균보다도 낮은 수준입니다.",market_commentary_v13,academy,qwen3.5-9b
4,653,2026-08-06,005930,삼성전자,BUY,0.337980,87,매우 높음,ready,56.0,31.0,66.0,4687,llm,ready,"현재는 개인투자자의 매수가 더 많습니다. 댓글 수급 신호가 87점으로 매우 높은 편이며, 이는 최근 5거래일 평균인 66점보다 높습니다. 다만 어제와 비교하면 31점이 높아졌습니다.","삼성전자의 현재 댓글 수급 신호는 87점으로 매우 높은 수준이며, 직전 거래일 대비 31점 상승하고 최근 5거래일 평균 66점보다도 높은 상태입니다.",market_commentary_v13,academy,qwen3.5-9b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,578,2026-07-27,068270,셀트리온,SELL,-0.201158,85,매우 높음,ready,NaN,NaN,NaN,62,deterministic,insufficient_evidence,"셀트리온은 현재 개인투자자의 매도가 더 많고, 댓글 수급 신호는 85점으로 매우 높은 편입니다.","셀트리온은 현재 개인투자자의 매도가 더 많고, 댓글 수급 신호는 85점으로 매우 높은 편입니다.",market_commentary_v13,academy,qwen3.5-9b
86,577,2026-07-27,247540,에코프로비엠,BUY,0.122106,81,매우 높음,ready,NaN,NaN,NaN,12,deterministic,insufficient_evidence,"에코프로비엠은 현재 개인투자자의 매수가 더 많고, 댓글 수급 신호는 81점으로 매우 높은 편입니다.","에코프로비엠은 현재 개인투자자의 매수가 더 많고, 댓글 수급 신호는 81점으로 매우 높은 편입니다.",market_commentary_v13,academy,qwen3.5-9b
87,574,2026-07-27,035720,카카오,SELL,-0.104476,6,매우 낮음,ready,NaN,NaN,NaN,78,deterministic,insufficient_evidence,"카카오는 현재 개인투자자의 매도가 더 많고, 댓글 수급 신호는 6점으로 매우 낮은 편입니다.","카카오는 현재 개인투자자의 매도가 더 많고, 댓글 수급 신호는 6점으로 매우 낮은 편입니다.",market_commentary_v13,academy,qwen3.5-9b
88,580,2026-07-27,005490,포스코홀딩스,SELL,-0.083689,70,높음,ready,NaN,NaN,NaN,8,deterministic,insufficient_evidence,"포스코홀딩스는 현재 개인투자자의 매도가 더 많고, 댓글 수급 신호는 70점으로 높은 편입니다.","포스코홀딩스는 현재 개인투자자의 매도가 더 많고, 댓글 수급 신호는 70점으로 높은 편입니다.",market_commentary_v13,academy,qwen3.5-9b


## 5. 정형 근거 검수

LLM에 실제로 전달한 `details.evidence`를 펼친다. 이 값이 LLM 입력의
전부이며 키워드와 댓글 원문은 포함되지 않는다.


In [7]:
evidence_rows = []

for row_index, report in enumerate(report_series):
    base = base_df.iloc[row_index]
    details = report.get("details", {})
    evidence = details.get("evidence", {})

    evidence_rows.append(
        {
            "보고서ID": base["llm_report_id"],
            "기준일": report.get("model_date"),
            "종목코드": report.get("stock_code"),
            "종목명": report.get("stock_name"),
            "활성모델": details.get("active_model_variant"),
            "raw예측값": details.get("predicted_score"),
            "인식특성수": details.get("recognized_feature_count"),
            "수급방향": evidence.get("supply_direction"),
            "실제수급지수": evidence.get("actual_supply_index"),
            "신호상태": evidence.get("signal_status"),
            "댓글신호": evidence.get("comment_signal_score"),
            "신호강도": evidence.get("signal_level"),
            "전일신호": evidence.get("previous_signal_score"),
            "전일대비": evidence.get("signal_change"),
            "직전5일평균": evidence.get("signal_ma5"),
            "댓글수": evidence.get("comment_count"),
            "calibration버전": details.get("calibration_schema_version"),
        }
    )

signal_evidence_df = pd.DataFrame(evidence_rows).sort_values(
    by=["기준일", "종목명"],
    ascending=[False, True],
).reset_index(drop=True)

signal_evidence_df


,보고서ID,기준일,종목코드,종목명,활성모델,raw예측값,인식특성수,수급방향,실제수급지수,신호상태,댓글신호,신호강도,전일신호,전일대비,직전5일평균,댓글수,calibration버전
0,656,2026-08-06,373220,LG에너지솔루션,negative,-0.201719,20,SELL,-0.532216,ready,70,높음,83.0,-13.0,80.0,9,1
1,655,2026-08-06,035420,NAVER,negative,-0.113441,410,SELL,-0.150678,ready,21,낮음,65.0,-44.0,51.0,151,1
2,652,2026-08-06,000660,SK하이닉스,positive,0.229751,5437,BUY,0.439048,ready,75,높음,53.0,22.0,59.0,9702,1
3,661,2026-08-06,034020,두산에너빌리티,positive,-0.013544,449,BUY,0.062034,ready,1,매우 낮음,70.0,-69.0,48.0,231,1
4,653,2026-08-06,005930,삼성전자,positive,0.278655,4106,BUY,0.337980,ready,87,매우 높음,56.0,31.0,66.0,4687,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,578,2026-07-27,068270,셀트리온,negative,-0.247466,189,SELL,-0.201158,ready,85,매우 높음,NaN,NaN,NaN,62,1
86,577,2026-07-27,247540,에코프로비엠,positive,0.251010,87,BUY,0.122106,ready,81,매우 높음,NaN,NaN,NaN,12,1
87,574,2026-07-27,035720,카카오,negative,-0.066887,243,SELL,-0.104476,ready,6,매우 낮음,NaN,NaN,NaN,78,1
88,580,2026-07-27,005490,포스코홀딩스,negative,-0.201245,137,SELL,-0.083689,ready,70,높음,NaN,NaN,NaN,8,1


## 6. 계약 위반 점검

저장된 보고서를 **생산 코드와 동일한 검증기**로 다시 검사한다.

`report_json.details`에 재현 정보가 모두 들어 있고 `provider`·`model`은
`llm_report` 컬럼에 있으므로, 저장된 것만으로 `ReportGenerationRequest`를
복원할 수 있다. 검수용으로 규칙을 다시 구현하지 않는다.

`prompt_version`이 현재 코드와 다른 과거 행은 DTO 복원이 막힌다. 당시
계약으로 통과한 보고서를 지금 규칙으로 재단하면 판단이 흐려지므로
의도된 제약이며, 이 경우 재검증을 건너뛴다.


In [8]:
from datetime import date as date_type

from pilos.analysis.llm_report import validate_market_commentary_response
from pilos.dto.llm_report_dto import (
    LlmMarketCommentary,
    LlmSignalEvidence,
    LlmSupplyState,
    ReportGenerationRequest,
)


def restore_request(report: dict, base_row) -> ReportGenerationRequest:
    """저장된 보고서와 DB 컬럼으로 검증 요청을 복원한다."""
    details = report["details"]
    return ReportGenerationRequest(
        daily_document_id=report["daily_document_id"],
        positive_result_id=details["positive_result_id"],
        negative_result_id=details["negative_result_id"],
        stock_id=report["stock_id"],
        stock_code=report["stock_code"],
        stock_name=report["stock_name"],
        model_date=date_type.fromisoformat(report["model_date"]),
        comment_count=report["comment_count"],
        supply_state=LlmSupplyState(**report["supply_state"]),
        active_model_variant=details["active_model_variant"],
        predicted_score=details["predicted_score"],
        recognized_feature_count=details["recognized_feature_count"],
        evidence=LlmSignalEvidence(**details["evidence"]),
        model_name=details["model_name"],
        model_version=details["model_version"],
        artifact_schema_version=details["artifact_schema_version"],
        calibration_schema_version=details["calibration_schema_version"],
        provider=base_row["provider"],
        model=base_row["model"],
    )


violations = []
skipped = 0

for row_index, report in enumerate(report_series):
    base = base_df.iloc[row_index]

    if report.get("prompt_version") != PROMPT_VERSION:
        skipped += 1
        continue

    try:
        request = restore_request(report, base)
    except Exception as error:
        violations.append(
            {
                "보고서ID": base["llm_report_id"],
                "기준일": report.get("model_date"),
                "종목명": report.get("stock_name"),
                "작성주체": report.get("commentary_source"),
                "구분": "복원 실패",
                "사유": f"{type(error).__name__}: {error}",
            }
        )
        continue

    try:
        validate_market_commentary_response(
            request=request,
            response=LlmMarketCommentary(
                market_commentary=report["market_commentary"],
                conclusion=report["conclusion"],
            ),
        )
    except ValueError as error:
        violations.append(
            {
                "보고서ID": base["llm_report_id"],
                "기준일": report.get("model_date"),
                "종목명": report.get("stock_name"),
                "작성주체": report.get("commentary_source"),
                "구분": "검증 위반",
                "사유": str(error),
                "시장코멘터리": report["market_commentary"],
                "한줄정리": report["conclusion"],
            }
        )

violation_df = pd.DataFrame(violations)

if skipped:
    print(f"다른 prompt_version이라 재검증을 건너뛴 보고서: {skipped}건")

if violation_df.empty:
    print(f"계약 위반 없음 ({len(report_series) - skipped}건 재검증)")

violation_df


계약 위반 없음 (90건 재검증)


""


## 7. 검증에서 거부된 보고서

검증에 걸린 보고서는 DB에 저장되지 않는다. 실행 로그에만 남으므로
`logs/generate_llm_reports_*.log`에서 읽어온다.

적재된 보고서(통과분)와 거부된 보고서(실패분)를 함께 봐야 검증 규칙이
정말 필요했는지, 아니면 쓸 만한 보고서를 막았는지 판단할 수 있다.


In [9]:
import re

LOG_DIR = BASE_DIR / "logs"

# generate_llm_reports가 남기는 거부 기록 형식이다.
REJECTION_HEADER = re.compile(
    r"^(?P<logged_at>[\d\-]+ [\d:,]+) \| WARNING \| \S+ \| "
    r"LLM 응답 검증 거부: "
    r"stock_code=(?P<stock_code>[^,]*), "
    r"stock_name=(?P<stock_name>.*?), "
    r"model_date=(?P<model_date>[^,]*), "
    r"daily_document_id=(?P<daily_document_id>[^,]*), "
    r"시도=(?P<attempt>[^,]*), "
    r"supply_direction=(?P<supply_direction>[^,]*), "
    r"actual_supply_index=(?P<actual_supply_index>[^,]*), "
    r"recognized_feature_count=(?P<recognized_feature_count>[^,]*), "
    r"comment_signal_score=(?P<comment_signal_score>[^,]*), "
    r"previous_signal_score=(?P<previous_signal_score>[^,]*), "
    r"signal_change=(?P<signal_change>[^,]*), "
    r"signal_ma5=(?P<signal_ma5>[^,]*), "
    r"사유=(?P<reason>.*)$"
)
LOG_LINE_START = re.compile(r"^[\d]{4}-[\d]{2}-[\d]{2} [\d]{2}:[\d]{2}:[\d]{2}")


def parse_rejection_log(log_path) -> list[dict]:
    """실행 로그에서 거부 기록만 뽑아낸다."""
    records = []
    current = None
    field = None

    for raw_line in log_path.read_text(encoding="utf-8").splitlines():
        header = REJECTION_HEADER.match(raw_line)

        if header:
            current = header.groupdict()
            current["로그파일"] = log_path.name
            current["시장코멘터리"] = ""
            current["한줄정리"] = ""
            records.append(current)
            field = None
            continue

        if current is None:
            continue

        if raw_line.startswith("  [market_commentary] "):
            field = "시장코멘터리"
            current[field] = raw_line[len("  [market_commentary] "):]
        elif raw_line.startswith("  [conclusion] "):
            field = "한줄정리"
            current[field] = raw_line[len("  [conclusion] "):]
        elif field and not LOG_LINE_START.match(raw_line):
            # 본문이 여러 줄이면 이어 붙인다.
            current[field] = f"{current[field]}\n{raw_line}"
        else:
            current = None
            field = None

    return records


rejected_rows = []

if LOG_DIR.exists():
    for log_path in sorted(LOG_DIR.glob("generate_llm_reports_*.log")):
        rejected_rows.extend(parse_rejection_log(log_path))
else:
    print(f"로그 폴더가 없습니다: {LOG_DIR}")

rejected_df = pd.DataFrame(rejected_rows)

if rejected_df.empty:
    print("거부된 보고서 기록이 없습니다.")
else:
    rejected_df = rejected_df.rename(
        columns={
            "logged_at": "기록시각",
            "stock_code": "종목코드",
            "stock_name": "종목명",
            "model_date": "기준일",
            "daily_document_id": "일별문서ID",
            "attempt": "시도",
            "supply_direction": "수급방향",
            "actual_supply_index": "실제수급지수",
            "recognized_feature_count": "인식특성수",
            "comment_signal_score": "댓글신호",
            "previous_signal_score": "전일신호",
            "signal_change": "전일대비",
            "signal_ma5": "직전5일평균",
            "reason": "거부사유",
        }
    )
    rejected_df = rejected_df[
        [
            "기록시각",
            "기준일",
            "종목코드",
            "종목명",
            "시도",
            "수급방향",
            "댓글신호",
            "전일신호",
            "전일대비",
            "직전5일평균",
            "거부사유",
            "실제수급지수",
            "인식특성수",
            "시장코멘터리",
            "한줄정리",
            "일별문서ID",
            "로그파일",
        ]
    ].sort_values(
        ["기준일", "종목명", "시도"]
    ).reset_index(drop=True)
    print(f"거부 시도 {len(rejected_df)}건")
    print(rejected_df["시도"].value_counts().sort_index().to_string())

rejected_df


거부 시도 7건
시도
1    6
2    1


,기록시각,기준일,종목코드,종목명,시도,수급방향,댓글신호,전일신호,전일대비,직전5일평균,거부사유,실제수급지수,인식특성수,시장코멘터리,한줄정리,일별문서ID,로그파일
0,"2026-08-10 11:45:06,030",2026-07-28,247540,에코프로비엠,1,BUY,48,81,-33,81,"입력 signal_level을 다른 등급으로 표현했습니다: field=market_commentary, 입력=보통, 출력=['낮음']",0.20626396333085095,113,"현재는 개인투자자의 매수가 더 많습니다. 댓글 수급 신호는 48점으로 보통 수준이며, 직전 거래일의 81점에 비해 33점 낮아졌습니다. 다만, 이 점수는 최근 5거래일 평균인 81점에도 미치지 못하는 낮은 편입니다.","현재 개인투자자의 매수가 우세한 상태이나, 댓글 수급 신호는 어제보다 33점 낮아진 48점으로 최근 5일 평균인 81점에도 미치지 못하는 낮은 수준입니다.",15,generate_llm_reports_2026-08-10.log
1,"2026-08-10 11:44:25,370",2026-07-28,035720,카카오,1,SELL,19,6,13,6,"입력 signal_level을 다른 등급으로 표현했습니다: field=market_commentary, 입력=매우 낮음, 출력=['높음']",-0.0663034902816289,162,"현재는 개인투자자의 매도가 더 많은 상태입니다. 댓글 수급 신호가 19점으로 매우 낮은 편이나, 어제보다 13점 높아졌습니다. 다만 최근 5거래일 평균인 6점을 기준으로 보면 현재 점수가 더 높은 수준입니다.",어제보다 13점 높고 최근 5일 평균보다도 높은 19점으로 매우 낮은 상태,12,generate_llm_reports_2026-08-10.log
2,"2026-08-10 11:46:23,401",2026-07-29,005930,삼성전자,1,SELL,41,82,-41,82,수급 강도의 변화는 입력에 없습니다: '매도 압력이 더 강',-0.05744878333283,7383,"현재 삼성전자의 개인투자자 수급은 매도 압력이 더 강한 상황입니다. 오늘 댓글 수급 신호는 41점으로 보통 수준인데, 이는 어제 82점보다 41점 낮아진 수치입니다. 또한 최근 5거래일 평균인 82점과 비교해도 현재 점수는 낮은 편에 속합니다.","현재 개인투자자의 매도가 우세한 상태이며, 댓글 수급 신호는 41점으로 보통 수준입니다.",5413,generate_llm_reports_2026-08-10.log
3,"2026-08-10 11:46:42,147",2026-07-29,035720,카카오,1,SELL,50,19,31,13,"입력 signal_level을 다른 등급으로 표현했습니다: field=market_commentary, 입력=보통, 출력=['높음']",-0.19250168973268753,360,"현재는 개인투자자의 매도가 더 많습니다. 댓글 수급 신호가 50점으로 보통 수준인데, 어제보다 31점 높아졌습니다. 다만 최근 5거래일 평균인 13점을 넘긴 상태라 전체적으로 높은 편입니다.",어제보다 31점이 높고 최근 5거래일 평균보다도 높은 50점의 보통 수준을 유지하고 있습니다.,4,generate_llm_reports_2026-08-10.log
4,"2026-08-10 11:46:50,750",2026-07-29,035720,카카오,2,SELL,50,19,31,13,"평균 대비 비교 방향이 실제와 반대입니다: 현재=50, 평균=13, 본문='에는 아직 미치지 못했습니'",-0.19250168973268753,360,"현재는 개인투자자의 매도가 더 많습니다. 댓글 수급 신호가 50점으로 보통 수준입니다. 어제보다 31점 높아졌지만, 최근 5거래일 평균인 13점에는 아직 미치지 못했습니다.",어제보다 31점 높고 최근 5일 평균보다 높은 상태,4,generate_llm_reports_2026-08-10.log
5,"2026-08-10 11:47:42,897",2026-07-29,005380,현대차,1,SELL,23,90,-67,78,"입력 signal_level을 다른 등급으로 표현했습니다: field=conclusion, 입력=낮음, 출력=['매우 낮음']",-0.3499337275996315,1315,"현재는 개인투자자의 매도가 더 많습니다. 댓글 수급 신호는 23점으로 낮은 편이며, 이는 최근 5거래일 평균인 78점보다도 낮습니다. 다만 어제와 비교하면 67점이 크게 하락했습니다.","현재 현대차의 개인투자자 매도 압력이 우세한 가운데, 댓글 수급 신호가 23점으로 매우 낮은 수준을 유지하고 있습니다.",9,generate_llm_reports_2026-08-10.log
6,"2026-08-10 11:56:28,234",2026-08-05,247540,에코프로비엠,1,BUY,86,37,49,43,signal_ma5를 다른 통계처럼 표현했습니다: 과거 평균. '직전 5거래일 평균' 계열 표현을 쓰세요.,0.4874384564204923,263,"현재는 개인투자자의 매수가 더 많습니다. 댓글 수급 신호는 86점으로 매우 높은 편이며, 이는 직전 거래일의 37점보다 49포인트 크게 올랐습니다. 다만, 이 점수는 최근 5거래일 평균인 43점을 상회하고 있어 현재 수준이 과거 평균보다 뚜렷하게 높게 형성되고 있습니다.",에코프로비엠은 개인투자자의 매수가 우세한 가운데 댓글 신호가 매우 높은 86점으로 어제보다도 최근 5일 평균보다도 높은 상태입니다.,5356,generate_llm_reports_2026-08-10.log


### 거부 사유별 분포

한 사유가 몰려 있다면 그 규칙이 과한지 먼저 확인한다.


In [10]:
if rejected_df.empty:
    print("집계할 거부 기록이 없습니다.")
    rejection_summary_df = pd.DataFrame()
else:
    # 사유 문자열에서 규칙 이름만 떼어 묶는다. 같은 규칙이 숫자값
    # 때문에 다른 문자열로 갈라지지 않게 앞부분만 사용한다.
    normalized = rejected_df["거부사유"].str.split(":").str[0].str.strip()
    rejection_summary_df = (
        normalized.value_counts()
        .rename_axis("규칙")
        .reset_index(name="건수")
    )

    # 규칙 이름만으로는 판단이 어려우므로 실제 사유 한 건을 붙인다.
    examples = (
        rejected_df.assign(규칙=normalized)
        .groupby("규칙")["거부사유"]
        .first()
    )
    rejection_summary_df["예시사유"] = rejection_summary_df["규칙"].map(
        examples
    )

rejection_summary_df


,규칙,건수,예시사유
0,입력 signal_level을 다른 등급으로 표현했습니다,4,"입력 signal_level을 다른 등급으로 표현했습니다: field=market_commentary, 입력=보통, 출력=['낮음']"
1,수급 강도의 변화는 입력에 없습니다,1,수급 강도의 변화는 입력에 없습니다: '매도 압력이 더 강'
2,평균 대비 비교 방향이 실제와 반대입니다,1,"평균 대비 비교 방향이 실제와 반대입니다: 현재=50, 평균=13, 본문='에는 아직 미치지 못했습니'"
3,signal_ma5를 다른 통계처럼 표현했습니다,1,signal_ma5를 다른 통계처럼 표현했습니다: 과거 평균. '직전 5거래일 평균' 계열 표현을 쓰세요.


## 8. 상태별 요약


In [11]:
summary_df = (
    signal_evidence_df.groupby(["신호상태", "수급방향"], dropna=False)
    .agg(
        건수=("보고서ID", "count"),
        평균신호=("댓글신호", "mean"),
        최소신호=("댓글신호", "min"),
        최대신호=("댓글신호", "max"),
    )
    .reset_index()
)

summary_df


,신호상태,수급방향,건수,평균신호,최소신호,최대신호
0,ready,BUY,36,63.138889,1,96
1,ready,SELL,54,52.888889,2,96


## 9. 검수 산출물 저장

`data`는 Git 비추적 경로다. 저장 파일은 로컬 검수용으로만 사용한다.


In [12]:
review_dir = BASE_DIR / "data" / "review"
review_dir.mkdir(parents=True, exist_ok=True)

version_tag = REPORT_VERSION.replace("market_commentary_", "")


def save_csv(frame, name: str) -> None:
    if frame is None or frame.empty:
        print(f"저장 생략(내용 없음): {name}")
        return

    path = review_dir / name
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"저장: {path.name} ({len(frame)}행)")


save_csv(review_df, f"llm_report_review_{version_tag}.csv")
save_csv(
    signal_evidence_df,
    f"llm_report_signal_evidence_{version_tag}.csv",
)
save_csv(violation_df, f"llm_report_violations_{version_tag}.csv")
save_csv(rejected_df, f"llm_report_rejected_{version_tag}.csv")
save_csv(
    rejection_summary_df,
    f"llm_report_rejection_summary_{version_tag}.csv",
)

print(f"\n저장 경로: {review_dir}")


저장: llm_report_review_v13.csv (90행)
저장: llm_report_signal_evidence_v13.csv (90행)
저장 생략(내용 없음): llm_report_violations_v13.csv
저장: llm_report_rejected_v13.csv (7행)
저장: llm_report_rejection_summary_v13.csv (4행)

저장 경로: d:\PLIOS\pilos-sentiment-index\data\review
